# Step 2 — Week 3: Evaluation Rubrics, Benchmarks, and Repeatable Scoring

## Goals
- Design evaluation rubrics: correctness, faithfulness, format compliance
- Create a benchmark set with 30–50 prompts
- Measure response quality with repeatable scoring

## Prerequisites
- Ollama installed and running: `ollama serve`
- Model pulled: `ollama pull llama3.2`
- Dependencies: `pip install requests jsonschema pandas`

## Setup

In [11]:
import json
import random
import statistics
from typing import Any, Dict, List
import re
import pandas as pd
import requests
from jsonschema import validate, ValidationError

OLLAMA_URL = "http://localhost:11434"
MODEL = "llama3.2"

# For repeatable benchmark order/sampling
RANDOM_SEED = 42
random.seed(RANDOM_SEED)


def ollama_generate(
    prompt: str,
    system: str = "You are a helpful assistant.",
    temperature: float = 0.0,
    top_p: float = 0.9,
    model: str = MODEL,
) -> str:
    body = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "top_p": top_p,
        },
    }
    if system:
        body["system"] = system

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json=body,
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["response"].strip()

print("Setup complete.")
print("Model:", MODEL)
print("Seed:", RANDOM_SEED)

Setup complete.
Model: llama3.2
Seed: 42


---
## Part 1 — Evaluation Rubrics

We score each response on three dimensions (1–5):
- **Correctness**: factual/task accuracy
- **Faithfulness**: grounded in provided context (no unsupported claims)
- **Format compliance**: follows required structure exactly

In [2]:
RUBRIC = {
    "correctness": {
        "description": "How accurate and complete the answer is for the task.",
        "scale": {
            1: "Incorrect or irrelevant",
            2: "Mostly incorrect with minor correct pieces",
            3: "Partially correct, important gaps",
            4: "Mostly correct with small issues",
            5: "Fully correct and complete",
        },
    },
    "faithfulness": {
        "description": "How well the answer stays grounded in provided context.",
        "scale": {
            1: "Hallucinates heavily",
            2: "Several unsupported claims",
            3: "Some unsupported claims",
            4: "Mostly grounded, minor drift",
            5: "Fully grounded in context",
        },
    },
    "format_compliance": {
        "description": "How strictly the answer follows requested output format.",
        "scale": {
            1: "Wrong format",
            2: "Major format violations",
            3: "Partially follows format",
            4: "Mostly compliant",
            5: "Exactly compliant",
        },
    },
}

print("Rubric loaded with dimensions:", list(RUBRIC.keys()))

Rubric loaded with dimensions: ['correctness', 'faithfulness', 'format_compliance']


---
## Part 2 — Benchmark Set (40 prompts)

Each benchmark item contains:
- `id`
- `category`
- `prompt`
- `context` (optional)
- `format_requirement`

In [16]:
BENCHMARK_SET = [
    # Summarization (10)
    {"id": "S01", "category": "summarization", "prompt": "Summarize: Transformers use self-attention to model token relationships.", "context": "", "format_requirement": "one sentence"},
    {"id": "S02", "category": "summarization", "prompt": "Summarize this in 20 words: RAG combines retrieval with generation to improve factual grounding.", "context": "", "format_requirement": "20 words max"},
    
    # QA with context (10)
    {"id": "Q01", "category": "qa", "prompt": "Answer: What is the capital of France?", "context": "France's capital is Paris.", "format_requirement": "short answer"},
    {"id": "Q02", "category": "qa", "prompt": "Answer from context only: Which city is Japan's capital?", "context": "Japan's capital is Tokyo.", "format_requirement": "short answer"},
   

    # Format compliance tasks (10)
    {"id": "F01", "category": "format", "prompt": "Return ONLY JSON: {""sentiment"": ""POSITIVE|NEGATIVE|NEUTRAL""} for text: 'I love this book.'", "context": "", "format_requirement": "json"},
    {"id": "F02", "category": "format", "prompt": "Return exactly: YES or NO. Is Python interpreted?", "context": "", "format_requirement": "yes/no"},
   

    # Faithfulness stress tests (10)
    {"id": "H01", "category": "faithfulness", "prompt": "Using context only, what city hosted Expo 1900?", "context": "Paris hosted the 1900 Exposition Universelle.", "format_requirement": "short answer"},
    {"id": "H02", "category": "faithfulness", "prompt": "Answer from context only: What is model X's context window?", "context": "Model X context window is 8192 tokens.", "format_requirement": "short answer"},
]
print(f"Benchmark size: {len(BENCHMARK_SET)} prompts")
print("Categories:", sorted({item['category'] for item in BENCHMARK_SET}))

Benchmark size: 8 prompts
Categories: ['faithfulness', 'format', 'qa', 'summarization']


---
## Part 3 — Repeatable Scoring Pipeline

We use a deterministic judge prompt (`temperature=0.0`) and fixed benchmark order to make scores repeatable.

In [17]:
JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "correctness": {"type": "integer", "minimum": 1, "maximum": 5},
        "faithfulness": {"type": "integer", "minimum": 1, "maximum": 5},
        "format_compliance": {"type": "integer", "minimum": 1, "maximum": 5},
        "notes": {"type": "string"},
    },
    "required": ["correctness", "faithfulness", "format_compliance", "notes"],
    "additionalProperties": False,
}


def validate_json_output(data: Dict[str, Any], schema: Dict[str, Any]) -> tuple[bool, str | None]:
    try:
        validate(instance=data, schema=schema)
        return True, None
    except ValidationError as e:
        path = ".".join(str(p) for p in e.absolute_path)
        return False, f"Validation error at '{path}': {e.message}"
    except Exception as e:
        return False, f"Unexpected validation error: {e}"


def extract_json(response_text: str) -> Dict[str, Any] | None:
    match = re.search(r"\{[\s\S]*\}", response_text)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def judge_response(item: Dict[str, Any], model_output: str) -> Dict[str, Any]:
    judge_prompt = f"""You are an LLM evaluator.
Evaluate a model output on 3 dimensions: correctness, faithfulness, format_compliance.

Rubric:
- correctness: 1 (wrong) to 5 (fully correct)
- faithfulness: 1 (hallucinated) to 5 (fully grounded in context)
- format_compliance: 1 (violates format) to 5 (fully compliant)

Task Prompt:
{item['prompt']}

Context (if empty, treat as no context):
{item['context']}

Format Requirement:
{item['format_requirement']}

Model Output:
{model_output}

Return ONLY valid JSON:
{{
  "correctness": 1,
  "faithfulness": 1,
  "format_compliance": 1,
  "notes": "short reason"
}}"""

    raw = ollama_generate(judge_prompt, temperature=0.0)
    data = extract_json(raw)

    if data is None:
        return {
            "correctness": 1,
            "faithfulness": 1,
            "format_compliance": 1,
            "notes": "judge output was not valid JSON",
        }

    is_valid, error = validate_json_output(data, JUDGE_SCHEMA)
    if not is_valid:
        return {
            "correctness": 1,
            "faithfulness": 1,
            "format_compliance": 1,
            "notes": f"judge schema invalid: {error}",
        }

    return data


def run_benchmark(
    benchmark_items: List[Dict[str, Any]],
    answer_temperature: float = 0.0,
    max_items: int | None = None,
) -> pd.DataFrame:
    items = benchmark_items if max_items is None else benchmark_items[:max_items]
    rows: List[Dict[str, Any]] = []

    for idx, item in enumerate(items, start=1):
        print(f"[{idx}/{len(items)}] Running {item['id']} ({item['category']})")

        user_prompt = item["prompt"]
        if item["context"]:
            user_prompt = f"Context:\n{item['context']}\n\nTask:\n{item['prompt']}"

        model_output = ollama_generate(user_prompt, temperature=answer_temperature)
        scores = judge_response(item, model_output)

        rows.append(
            {
                "id": item["id"],
                "category": item["category"],
                "format_requirement": item["format_requirement"],
                "prompt": item["prompt"],
                "context": item["context"],
                "model_output": model_output,
                "correctness": scores["correctness"],
                "faithfulness": scores["faithfulness"],
                "format_compliance": scores["format_compliance"],
                "notes": scores["notes"],
                "total_score": scores["correctness"] + scores["faithfulness"] + scores["format_compliance"],
            }
        )

    return pd.DataFrame(rows)


print("Repeatable scoring pipeline ready.")

Repeatable scoring pipeline ready.


---
## Part 4 — Run Benchmark and Measure Quality

Start with a small run (`max_items=8`) for speed, then run full benchmark (`max_items=None`).

In [18]:
# Quick run for iteration speed
results_df = run_benchmark(BENCHMARK_SET, answer_temperature=0.0, max_items=None)

print("\nSample results:")
display(results_df[["id", "category", "correctness", "faithfulness", "format_compliance", "total_score"]].head(8))

print("\nAverage scores (quick run):")
print(results_df[["correctness", "faithfulness", "format_compliance", "total_score"]].mean().round(2))

[1/8] Running S01 (summarization)
[2/8] Running S02 (summarization)
[3/8] Running Q01 (qa)
[4/8] Running Q02 (qa)
[5/8] Running F01 (format)
[6/8] Running F02 (format)
[7/8] Running H01 (faithfulness)
[8/8] Running H02 (faithfulness)

Sample results:


,id,category,correctness,faithfulness,format_compliance,total_score
0,S01,summarization,1,1,1,3
1,S02,summarization,5,4,5,14
2,Q01,qa,5,5,5,15
3,Q02,qa,5,5,5,15
4,F01,format,5,5,5,15
5,F02,format,3,5,5,13
6,H01,faithfulness,5,5,5,15
7,H02,faithfulness,4,5,5,14



Average scores (quick run):
correctness           4.12
faithfulness          4.38
format_compliance     4.50
total_score          13.00
dtype: float64


In [20]:
def summarize_results(df: pd.DataFrame) -> None:
    print("Overall averages:")
    print(df[["correctness", "faithfulness", "format_compliance", "total_score"]].mean().round(2))

    print("\nCategory averages:")
    category_means = (
        df.groupby("category")[["correctness", "faithfulness", "format_compliance", "total_score"]]
        .mean()
        .round(2)
        .sort_values("total_score", ascending=False)
    )
    display(category_means)

    print("\nLowest-scoring examples:")
    cols = ["id", "category", "total_score", "correctness", "faithfulness", "format_compliance", "notes"]
    display(df.sort_values("total_score", ascending=True)[cols].head(5))


summarize_results(results_df)

Overall averages:
correctness           4.12
faithfulness          4.38
format_compliance     4.50
total_score          13.00
dtype: float64

Category averages:


,correctness,faithfulness,format_compliance,total_score
category,,,,
qa,5.0,5.0,5.0,15.0
faithfulness,4.5,5.0,5.0,14.5
format,4.0,5.0,5.0,14.0
summarization,3.0,2.5,3.0,8.5



Lowest-scoring examples:


,id,category,total_score,correctness,faithfulness,format_compliance,notes
0,S01,summarization,3,1,1,1,judge output was not valid JSON
5,F02,format,13,3,5,5,"The answer 'YES' is correct, but it's a yes/no..."
7,H02,faithfulness,14,4,5,5,"The model output is fully correct, faithful to..."
1,S02,summarization,14,5,4,5,Model accurately summarizes RAG's purpose and ...
3,Q02,qa,15,5,5,5,"Fully correct, faithful to context, and fully ..."
